In [2]:
pip install -U jupyterlab_widgets

Note: you may need to restart the kernel to use updated packages.


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import GPT2LMHeadModel, GPT2Config

torch.manual_seed(1337)

In [7]:
model = GPT2LMHeadModel.from_pretrained("gpt2")

print(model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)


In [8]:
config = model.config

print(config)

GPT2Config {
  "activation_function": "gelu_new",
  "add_cross_attention": false,
  "architectures": [
    "GPT2LMHeadModel"
  ],
  "attn_pdrop": 0.1,
  "bos_token_id": 50256,
  "dtype": "float32",
  "embd_pdrop": 0.1,
  "eos_token_id": 50256,
  "initializer_range": 0.02,
  "layer_norm_epsilon": 1e-05,
  "model_type": "gpt2",
  "n_ctx": 1024,
  "n_embd": 768,
  "n_head": 12,
  "n_inner": null,
  "n_layer": 12,
  "n_positions": 1024,
  "pad_token_id": null,
  "reorder_and_upcast_attn": false,
  "resid_pdrop": 0.1,
  "scale_attn_by_inverse_layer_idx": false,
  "scale_attn_weights": true,
  "summary_activation": null,
  "summary_first_dropout": 0.1,
  "summary_proj_to_labels": true,
  "summary_type": "cls_index",
  "summary_use_proj": true,
  "task_specific_params": {
    "text-generation": {
      "do_sample": true,
      "max_length": 50
    }
  },
  "tie_word_embeddings": true,
  "transformers_version": "5.5.4",
  "use_cache": true,
  "vocab_size": 50257
}



In [9]:
print("Vocabulary size :", config.vocab_size)
print("Context length  :", config.n_positions)
print("Embedding dim   :", config.n_embd)
print("Layers          :", config.n_layer)
print("Attention heads :", config.n_head)

Vocabulary size : 50257
Context length  : 1024
Embedding dim   : 768
Layers          : 12
Attention heads : 12


In [10]:
num_params = sum(p.numel() for p in model.parameters())

print(f"Total parameters: {num_params:,}")
print(f"Total parameters: {num_params / 1e6:.2f}M")

Total parameters: 124,439,808
Total parameters: 124.44M


In [11]:
for name, module in model.named_modules():
    if name:
        print(name, "->", module.__class__.__name__)

transformer -> GPT2Model
transformer.wte -> Embedding
transformer.wpe -> Embedding
transformer.drop -> Dropout
transformer.h -> ModuleList
transformer.h.0 -> GPT2Block
transformer.h.0.ln_1 -> LayerNorm
transformer.h.0.attn -> GPT2Attention
transformer.h.0.attn.c_attn -> Conv1D
transformer.h.0.attn.c_proj -> Conv1D
transformer.h.0.attn.attn_dropout -> Dropout
transformer.h.0.attn.resid_dropout -> Dropout
transformer.h.0.ln_2 -> LayerNorm
transformer.h.0.mlp -> GPT2MLP
transformer.h.0.mlp.c_fc -> Conv1D
transformer.h.0.mlp.c_proj -> Conv1D
transformer.h.0.mlp.act -> NewGELUActivation
transformer.h.0.mlp.dropout -> Dropout
transformer.h.1 -> GPT2Block
transformer.h.1.ln_1 -> LayerNorm
transformer.h.1.attn -> GPT2Attention
transformer.h.1.attn.c_attn -> Conv1D
transformer.h.1.attn.c_proj -> Conv1D
transformer.h.1.attn.attn_dropout -> Dropout
transformer.h.1.attn.resid_dropout -> Dropout
transformer.h.1.ln_2 -> LayerNorm
transformer.h.1.mlp -> GPT2MLP
transformer.h.1.mlp.c_fc -> Conv1D
tran

In [12]:
block = model.transformer.h[0]

print(block)

GPT2Block(
  (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (attn): GPT2Attention(
    (c_attn): Conv1D(nf=2304, nx=768)
    (c_proj): Conv1D(nf=768, nx=768)
    (attn_dropout): Dropout(p=0.1, inplace=False)
    (resid_dropout): Dropout(p=0.1, inplace=False)
  )
  (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  (mlp): GPT2MLP(
    (c_fc): Conv1D(nf=3072, nx=768)
    (c_proj): Conv1D(nf=768, nx=3072)
    (act): NewGELUActivation()
    (dropout): Dropout(p=0.1, inplace=False)
  )
)


In [14]:
print("Token embedding:", model.transformer.wte.weight.shape)
print("Position embedding:", model.transformer.wpe.weight.shape)

print("Attention QKV:", block.attn.c_attn.weight.shape)
print("Attention output:", block.attn.c_proj.weight.shape)

print("MLP first layer:", block.mlp.c_fc.weight.shape)
print("MLP second layer:", block.mlp.c_proj.weight.shape)

Token embedding: torch.Size([50257, 768])
Position embedding: torch.Size([1024, 768])
Attention QKV: torch.Size([768, 2304])
Attention output: torch.Size([768, 768])
MLP first layer: torch.Size([768, 3072])
MLP second layer: torch.Size([3072, 768])


In [16]:
print(reference_model.transformer.wte)
print(reference_model.transformer.wpe)

print("token embeddings :", reference_model.transformer.wte.weight.shape)
print("position embeddings:", reference_model.transformer.wpe.weight.shape)

Embedding(50257, 768)
Embedding(1024, 768)
token embeddings : torch.Size([50257, 768])
position embeddings: torch.Size([1024, 768])


In [17]:
class GPT2Embeddings(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd):
        super().__init__()

        self.wte = nn.Embedding(vocab_size, n_embd)
        self.wpe = nn.Embedding(block_size, n_embd)

    def forward(self, idx):
        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        tok_emb = self.wte(idx)
        pos_emb = self.wpe(positions)

        return tok_emb + pos_emb

In [19]:
embeddings = GPT2Embeddings(
    config.vocab_size,
    config.n_positions,
    config.n_embd,
)

idx = torch.randint(
    0,
    config.vocab_size,
    (2, 8),
)

x = embeddings(idx)

print("input :", idx.shape)
print("output:", x.shape)

input : torch.Size([2, 8])
output: torch.Size([2, 8, 768])


In [20]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True):
        super().__init__()

        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5,
        )

In [21]:
ln = LayerNorm(config.n_embd)

x = torch.randn(2, 8, config.n_embd)
y = ln(x)

print("input :", x.shape)
print("output:", y.shape)
print("mean  :", y.mean().item())
print("std   :", y.std().item())

input : torch.Size([2, 8, 768])
output: torch.Size([2, 8, 768])
mean  : 0.0
std   : 1.0000356435775757
